## DS-2002 Data Project 2: Building a Data Lakehouse with PySpark Structured Streaming

**Business Process**: Music Store Sales & Inventory Analysis (Chinook Database)

This project demonstrates the design and implementation of a dimensional Data Lakehouse using the Medallion Architecture (Bronze-Silver-Gold layers) with PySpark Structured Streaming. The system models a music retail business process, tracking invoice transactions and enabling post-hoc analysis of sales performance across multiple dimensions including customers, tracks, albums, artists, genres, and time periods.

### Project Overview

**Dimensional Model Components:**
- **Fact Table**: Invoice line items (sales transactions)
- **Dimension Tables**: 
  - `dim_date` - Enables temporal analysis with fiscal and calendar periods
  - `dim_customers` - Customer demographics and location
  - `dim_track` - Music tracks with album, artist, and genre hierarchies
  - `dim_employees` - Support representatives
  - `dim_playlists` - Music playlist categorization

**Data Integration Pattern**: ELTL (Extract-Load-Transform-Load)
- **Extract**: Source data from MySQL (dimensions) and MongoDB (reference data)
- **Load**: Ingest streaming JSON invoice data into Bronze layer
- **Transform**: Join with dimensions and calculate metrics in Silver layer
- **Load**: Aggregate business metrics in Gold layer for analytics

**Architecture**: Kappa Architecture (streaming-first approach)
- All data treated as streams (batch data processed as finite streams)
- Bronze layer: Raw data ingestion from streaming sources
- Silver layer: Integration with reference data from multiple sources
- Gold layer: Business-ready aggregations and analytics

### Data Sources (Multi-Source Integration)
1. **Relational Database (MySQL)**: Dimension tables (customers, employees, date dimension)
2. **NoSQL Database (MongoDB Atlas)**: Reference data (tracks, albums, artists, genres, media types, playlists)
3. **File System (JSON Streaming)**: Real-time invoice transaction data

### Key Technologies
- **Relational Database Management Systems**: MySQL (OLAP dimensional model)
- **NoSQL Systems**: MongoDB Atlas (JSON document storage)
- **File System Data Lake**: Parquet format for efficient columnar storage
- **Massively Parallel Processing**: Apache Spark/PySpark for distributed processing
- **Streaming Integration**: PySpark Structured Streaming with AutoLoader pattern

### Business Value Demonstration
The Gold layer provides actionable analytics:
1. **Top Tracks by Revenue**: Identifies best-selling products for inventory optimization
2. **Genre Performance by Fiscal Quarter**: Enables strategic planning and trend analysis
3. **Customer Engagement Metrics**: Tracks unique customers per genre/period

---

## Section I: Prerequisites

### 1.0. Import Required Libraries

In [7]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/anaconda3/envs/ds2002/lib/python3.12/site-packages/pyspark


### 2.0. Instantiate Global Variables

In [8]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "chinook_dw2",
    "conn_props" : {
        "user" : "root",
        "password" : "Huekmibap3224!",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}


# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "user_name" : "gft3fr",
    "password" : "Huekmibap3224!",
    "cluster_name" : "ds2002-lab4",
    "cluster_subnet" : "kapfjei",
    "cluster_location" : "atlas", # "local"
    "db_name" : "chinook",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'project_data')
data_dir = os.path.join(base_dir, 'chinook')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

invoices_stream_dir = os.path.join(stream_dir, 'orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "chinook_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

invoices_output_bronze = os.path.join(database_dir, 'fact_invoices', 'bronze')
invoices_output_silver = os.path.join(database_dir, 'fact_invoices', 'silver')
invoices_output_gold = os.path.join(database_dir, 'fact_invoices', 'gold')

### 3.0. Define Global Functions

In [9]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Chinook Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
    .set('spark.mongodb.input.uri', args['mongo_uri']) \
    .set('spark.mongodb.output.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    '''Query MongoDB, and create a DataFrame'''
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()

    '''Drop the '_id' index column to clean up the response.'''
    dframe = dframe.drop('_id')
    
    '''Call the drop_null_columns() function passing in the dataframe.'''
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

### 4.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory Structure to Ensure Idempotency

In [10]:
remove_directory_tree(database_dir)

"Directory '/Users/andrewotwell/Documents/DS-2002/project-2/spark-warehouse/chinook_dlh.db' has been removed successfully."

### 5.0. Create a New Spark Session

In [ ]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")

jars.append(mysql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

### 6.0. Create a New Metadata Database.

In [12]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'Data Lakehouse for DS-2002 Project 2'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Project 2');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions by Ingesting "Cold-path" Reference Data 
### 1.0. Fetch Data from the File System
#### 1.1. Verify the location of the source data files on the file system

In [13]:
get_file_info(batch_dir)

,name,size,modification_time
0,Album.json,30525,2025-12-20 04:26:58.299771309
1,Artist.json,17963,2025-12-20 04:26:58.290911674
2,Customer.json,20966,2025-12-20 04:26:58.287075996
3,Employee.json,3594,2025-12-20 04:59:15.074624062
4,Genre.json,1293,2025-12-20 04:26:58.273299932
5,MediaType.json,337,2025-12-20 04:26:58.234877348
6,Playlist.json,1041,2025-12-20 05:00:25.781960726
7,Track.json,813283,2025-12-20 04:26:58.231047630


#### 1.2. Populate the <span style="color:darkred">Tracks Dimension</span>
##### 1.2.1. Use PySpark to Read data from JSON files

In [14]:
# Read tracks JSON file
tracks_json = os.path.join(batch_dir, 'Track.json')
print(tracks_json)

df_tracks = spark.read.option("multiLine", "true").json(tracks_json)
df_tracks.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/Track.json


,AlbumId,Bytes,Composer,GenreId,MediaTypeId,Milliseconds,Name,TrackId,UnitPrice
0,1,11170334,"Angus Young, Malcolm Young, Brian Johnson",1,1,343719,For Those About To Rock (We Salute You),1,0.99
1,2,5510424,None,1,2,342562,Balls to the Wall,2,0.99


In [15]:
# Read albums JSON file
albums_json = os.path.join(batch_dir, 'Album.json')
print(albums_json)

df_albums = spark.read.option("multiLine", "true").json(albums_json)
df_albums.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/Album.json


,AlbumId,ArtistId,Title
0,1,1,For Those About To Rock We Salute You
1,2,2,Balls to the Wall


In [16]:
# Read artists JSON file
artists_json = os.path.join(batch_dir, 'Artist.json')
print(artists_json)

df_artists = spark.read.option("multiLine", "true").json(artists_json)
df_artists.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/Artist.json


,ArtistId,Name
0,1,AC/DC
1,2,Accept


In [17]:
# Read genres JSON file
genres_json = os.path.join(batch_dir, 'Genre.json')
print(genres_json)

df_genres = spark.read.option("multiLine", "true").json(genres_json)
df_genres.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/Genre.json


,GenreId,Name
0,1,Rock
1,2,Jazz


In [18]:
# Read media types JSON file
mediatypes_json = os.path.join(batch_dir, 'MediaType.json')
print(mediatypes_json)

df_mediatypes = spark.read.option("multiLine", "true").json(mediatypes_json)
df_mediatypes.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/MediaType.json


,MediaTypeId,Name
0,1,MPEG audio file
1,2,Protected AAC audio file


##### 1.2.2. Make Necessary Transformations to the New DataFrame

In [19]:
# ----------------------------------------------------------------------------------
# Rename columns for consistency
# ----------------------------------------------------------------------------------
df_tracks = (
    df_tracks
    .withColumnRenamed("TrackId", "track_id")
    .withColumnRenamed("Name", "track_name")
    .withColumnRenamed("AlbumId", "album_id")
    .withColumnRenamed("GenreId", "genre_id")
    .withColumnRenamed("MediaTypeId", "media_type_id")
    .withColumnRenamed("UnitPrice", "unit_price")
)

df_albums = (
    df_albums
    .withColumnRenamed("Title", "album_title")
    .withColumnRenamed("AlbumId", "album_id")
    .withColumnRenamed("ArtistId", "artist_id")
)

df_artists = (
    df_artists
    .withColumnRenamed("Name", "artist_name")
    .withColumnRenamed("ArtistId", "artist_id")
)

df_genres = (
    df_genres
    .withColumnRenamed("Name", "genre_name")
    .withColumnRenamed("GenreId", "genre_id")
)

df_mediatypes = (
    df_mediatypes
    .withColumnRenamed("Name", "media_type")
    .withColumnRenamed("MediaTypeId", "media_type_id")
)

In [20]:
# ----------------------------------------------------------------------------------
# Join all related tables to create comprehensive track dimension
# ----------------------------------------------------------------------------------
df_dim_track = ( 
    df_tracks.join(df_albums, "album_id", "left") 
    .join(df_artists, "artist_id", "left") 
    .join(df_genres, "genre_id", "left")
    .join(df_mediatypes, "media_type_id", "left")
)

In [21]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_track.createOrReplaceTempView("tracks")

sql_dim_track = """
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY track_id) AS track_key
    FROM tracks
"""

df_dim_track = spark.sql(sql_dim_track)

# ----------------------------------------------------------------------------------
# Rename/Reorder Columns and display the first two rows
# ----------------------------------------------------------------------------------
ordered_columns = [
    "track_key", "track_id", "track_name", "album_title", "artist_name", "genre_name", "media_type", "unit_price"]

df_dim_track = df_dim_track[ordered_columns]
df_dim_track.toPandas().head(2)

,track_key,track_id,track_name,album_title,artist_name,genre_name,media_type,unit_price
0,1,1,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
1,2,2,Balls to the Wall,Balls to the Wall,Accept,Rock,Protected AAC audio file,0.99


##### 1.2.3. Save as the <span style="color:darkred">dim_track</span> table in the Data Lakehouse

In [22]:
df_dim_track.write.saveAsTable(f"{dest_database}.dim_track", mode="overwrite")

##### 1.2.4. Unit Test: Describe and Preview Table

In [23]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_track;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_track LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           track_key|                 int|   NULL|
|            track_id|              bigint|   NULL|
|          track_name|              string|   NULL|
|         album_title|              string|   NULL|
|         artist_name|              string|   NULL|
|          genre_name|              string|   NULL|
|          media_type|              string|   NULL|
|          unit_price|              double|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|         chinook_dlh|       |
|               Table|           dim_track|       |
|        Created Time|Sat Dec 20 02:04:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.7|       |
|           

,track_key,track_id,track_name,album_title,artist_name,genre_name,media_type,unit_price
0,1,1,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
1,2,2,Balls to the Wall,Balls to the Wall,Accept,Rock,Protected AAC audio file,0.99


#### 1.3. Populate the <span style="color:darkred">Employees Dimension</span>
##### 1.3.1. Use PySpark to Read Data from JSON File

In [24]:
employees_json = os.path.join(batch_dir, 'Employee.json')
print(employees_json)

df_employees = spark.read.option("multiLine", "true").json(employees_json)
df_employees.toPandas().head(2)

/Users/andrewotwell/Documents/DS-2002/project-2/project_data/chinook/batch/Employee.json


,Address,BirthDate,City,Country,Email,EmployeeId,Fax,FirstName,HireDate,LastName,Phone,PostalCode,ReportsTo,State,Title
0,11120 Jasper Ave NW,1962-02-18 00:00:00,Edmonton,Canada,andrew@chinookcorp.com,1,+1 (780) 428-3457,Andrew,2002-08-14 00:00:00,Adams,+1 (780) 428-9482,T5K 2N1,NaN,AB,General Manager
1,825 8 Ave SW,1958-12-08 00:00:00,Calgary,Canada,nancy@chinookcorp.com,2,+1 (403) 262-3322,Nancy,2002-05-01 00:00:00,Edwards,+1 (403) 262-3443,T2P 2T3,1.0,AB,Sales Manager


##### 1.3.2 Make Necessary Transformations to the New DataFrame

In [25]:
# ----------------------------------------------------------------------------------
# Rename the columns
# ----------------------------------------------------------------------------------
df_dim_employees = (
    df_employees
    .withColumnRenamed("EmployeeId", "employee_id")
    .withColumnRenamed("LastName", "last_name")
    .withColumnRenamed("FirstName", "first_name")
    .withColumnRenamed("Title", "title")
    .withColumnRenamed("ReportsTo", "reports_to")
    .withColumnRenamed("BirthDate", "birth_date")
    .withColumnRenamed("HireDate", "hire_date")
    .withColumnRenamed("Address", "address")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("PostalCode", "postal_code")
    .withColumnRenamed("Phone", "phone")
    .withColumnRenamed("Fax", "fax")
    .withColumnRenamed("Email", "email")
)

# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY employee_id) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

# ----------------------------------------------------------------------------------
# Reorder Columns
# ----------------------------------------------------------------------------------
ordered_columns = [
    'employee_key', 'employee_id', 'first_name', 'last_name', 'title', 'reports_to',
    'birth_date', 'hire_date', 'phone', 'fax', 'email', 'address', 'city', 'state', 'country', 'postal_code'
]

df_dim_employees = df_dim_employees[ordered_columns]
df_dim_employees.toPandas().head(2)

,employee_key,employee_id,first_name,last_name,title,reports_to,birth_date,hire_date,phone,fax,email,address,city,state,country,postal_code
0,1,1,Andrew,Adams,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1
1,2,2,Nancy,Edwards,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3


##### 1.3.3. Save as the <span style="color:darkred">dim_employees</span> table in the Data Lakehouse

In [26]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employees", mode="overwrite")

##### 1.3.4. Unit Test: Describe and Preview Table

In [27]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employees LIMIT 2").toPandas()

+--------------------+-------------+-------+
|            col_name|    data_type|comment|
+--------------------+-------------+-------+
|        employee_key|          int|   NULL|
|         employee_id|       bigint|   NULL|
|          first_name|       string|   NULL|
|           last_name|       string|   NULL|
|               title|       string|   NULL|
|          reports_to|       bigint|   NULL|
|          birth_date|       string|   NULL|
|           hire_date|       string|   NULL|
|               phone|       string|   NULL|
|                 fax|       string|   NULL|
|               email|       string|   NULL|
|             address|       string|   NULL|
|                city|       string|   NULL|
|               state|       string|   NULL|
|             country|       string|   NULL|
|         postal_code|       string|   NULL|
|                    |             |       |
|# Detailed Table ...|             |       |
|             Catalog|spark_catalog|       |
|         

,employee_key,employee_id,first_name,last_name,title,reports_to,birth_date,hire_date,phone,fax,email,address,city,state,country,postal_code
0,1,1,Andrew,Adams,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1
1,2,2,Nancy,Edwards,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3


### 2.0. Fetch Reference Data from a MongoDB Atlas Database
#### 2.1. Create a New MongoDB Database, and Load Each JSON File into a New MongoDB Collection

In [28]:
client = get_mongo_client(**mongodb_args)

json_files = {"customers" : "Customer.json",
              "playlists" : "Playlist.json"}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files)

#### 2.2. Populate the <span style="color:darkred">Customers Dimension</span>
##### 2.2.1. Fetch Data from the New MongoDB <span style="color:darkred">Customers</span> Collection

In [29]:
mongodb_args["collection"] = "customers"

df_dim_customers = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_customers.toPandas().head(2)

,Address,City,Country,CustomerId,Email,FirstName,LastName,Phone,PostalCode,State,SupportRepId
0,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,Brazil,1,luisg@embraer.com.br,Luís,Gonçalves,+55 (12) 3923-5555,12227-000,SP,3
1,Theodor-Heuss-Straße 34,Stuttgart,Germany,2,leonekohler@surfeu.de,Leonie,Köhler,+49 0711 2842222,70174,None,5


##### 2.2.2. Make Necessary Transformations to the New Dataframe

In [30]:
# ----------------------------------------------------------------------------------
# Rename columns
# ----------------------------------------------------------------------------------
df_dim_customers = (
    df_dim_customers
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("Address", "address")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("Email", "email")
    .withColumnRenamed("FirstName", "first_name")
    .withColumnRenamed("LastName", "last_name")
    .withColumnRenamed("Phone", "phone")
    .withColumnRenamed("PostalCode", "postal_code")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("SupportRepId", "support_rep_id")
)

# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

# ----------------------------------------------------------------------------------
# Reorder Columns
# ----------------------------------------------------------------------------------
ordered_columns = ['customer_key', 'customer_id', 'first_name', 'last_name',
                   'phone', 'email' , 'address', 'city', 'state', 'country',
                   'postal_code', 'support_rep_id']

df_dim_customers = df_dim_customers[ordered_columns]
df_dim_customers.toPandas().head(2)

,customer_key,customer_id,first_name,last_name,phone,email,address,city,state,country,postal_code,support_rep_id
0,1,1,Luís,Gonçalves,+55 (12) 3923-5555,luisg@embraer.com.br,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3
1,2,2,Leonie,Köhler,+49 0711 2842222,leonekohler@surfeu.de,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,5


##### 2.2.3. Save as the <span style="color:darkred">dim_customers</span> table in the Data lakehouse

In [31]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

##### 2.2.4. Unit Test: Describe and Preview Table

In [32]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        customer_key|                 int|   NULL|
|         customer_id|                 int|   NULL|
|          first_name|              string|   NULL|
|           last_name|              string|   NULL|
|               phone|              string|   NULL|
|               email|              string|   NULL|
|             address|              string|   NULL|
|                city|              string|   NULL|
|               state|              string|   NULL|
|             country|              string|   NULL|
|         postal_code|              string|   NULL|
|      support_rep_id|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|         chinook_dlh|       |
|           

,customer_key,customer_id,first_name,last_name,phone,email,address,city,state,country,postal_code,support_rep_id
0,1,1,Luís,Gonçalves,+55 (12) 3923-5555,luisg@embraer.com.br,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3
1,2,2,Leonie,Köhler,+49 0711 2842222,leonekohler@surfeu.de,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,5


#### 2.3. Populate the <span style="color:darkred">Playlists Dimension</span>
##### 2.3.1. Fetch Data from the New MongoDB <span style="color:darkred">Playlists</span> Collection

In [33]:
mongodb_args["collection"] = "playlists"

df_dim_playlists = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_playlists.toPandas().head(2)

,Name,PlaylistId
0,Music,1
1,Movies,2


##### 2.3.2. Make Necessary Transformations to the New Dataframe

In [34]:
# ----------------------------------------------------------------------------------
# Rename columns
# ----------------------------------------------------------------------------------
df_dim_playlists = (
    df_dim_playlists
    .withColumnRenamed("PlaylistId", "playlist_id")
    .withColumnRenamed("Name", "playlist_name")
)

# ----------------------------------------------------------------------------------
# Add Primary Key column
# ----------------------------------------------------------------------------------
df_dim_playlists.createOrReplaceTempView("playlists")
sql_playlists = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY playlist_id) AS playlist_key
    FROM playlists;
"""
df_dim_playlists = spark.sql(sql_playlists)

# ----------------------------------------------------------------------------------
# Reorder Columns
# ----------------------------------------------------------------------------------
ordered_columns = ['playlist_key', 'playlist_id', 'playlist_name']

df_dim_playlists = df_dim_playlists[ordered_columns]
df_dim_playlists.toPandas().head(2)

,playlist_key,playlist_id,playlist_name
0,1,1,Music
1,2,2,Movies


##### 2.3.3. Save as the <span style="color:darkred">dim_playlists</span> table in the Data lakehouse

In [35]:
df_dim_playlists.write.saveAsTable(f"{dest_database}.dim_playlists", mode="overwrite")

##### 2.3.4. Unit Test: Describe and Preview Table

In [36]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_playlists;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_playlists LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        playlist_key|                 int|   NULL|
|         playlist_id|                 int|   NULL|
|       playlist_name|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|         chinook_dlh|       |
|               Table|       dim_playlists|       |
|        Created Time|Sat Dec 20 02:04:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.7|       |
|                Type|             MANAGED|       |
|            Provider|             parquet|       |
|            Location|file:/Users/andre...|       |
+--------------------+--------------------+-------+



,playlist_key,playlist_id,playlist_name
0,1,1,Music
1,2,2,Movies


### 3.0. Fetch Reference Data from a MySQL Database
#### 3.1. Populate the <span style="color:darkred">Date Dimension</span>
##### 3.1.1 Fetch data from the <span style="color:darkred">dim_date</span> table in MySQL

In [37]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)
df_dim_date.toPandas().head(2)

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


##### 3.1.2. Save as the <span style="color:darkred">dim_date</span> table in the Data Lakehouse

In [38]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

##### 3.1.3. Unit Test: Describe and Preview Table

In [39]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year|      int|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year|      int|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+
only showing top

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


### 4.0. Verify Dimension Tables

In [40]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,chinook_dlh,dim_customers,False
1,chinook_dlh,dim_date,False
2,chinook_dlh,dim_employees,False
3,chinook_dlh,dim_playlists,False
4,chinook_dlh,dim_track,False
5,,customers,True
6,,employees,True
7,,playlists,True
8,,tracks,True


## Section III: Integrate Reference Data with Real-Time Data
### 5.0. Use PySpark Structured Streaming to Process (Hot Path) <span style="color:darkred">Invoice</span> Fact Data  
#### 5.1. Verify the location of the source data files on the file system

In [41]:
get_file_info(invoices_stream_dir)

,name,size,modification_time
0,chinook_invoices_01.json,260219,2025-12-20 04:27:44.830660582
1,chinook_invoices_02.json,262882,2025-12-20 04:27:44.823625565
2,chinook_invoices_03.json,263236,2025-12-20 04:27:44.818493843


#### 5.2. Create the Bronze Layer: Stage <span style="color:darkred">Invoice Fact table</span> Data
##### 5.2.1. Read "Raw" JSON file data into a Stream

In [42]:
df_invoices_bronze = (
    spark.readStream \
    .option("schemaLocation", invoices_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(invoices_stream_dir)
)

df_invoices_bronze.isStreaming

True

##### 5.2.2. Write the Streaming Data to a Parquet file

In [43]:
invoices_checkpoint_bronze = os.path.join(invoices_output_bronze, '_checkpoint')

invoices_bronze_query = (
    df_invoices_bronze
    # Add Current Timestamp and Input Filename columns for Traceability
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("invoices_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", invoices_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(invoices_output_bronze)
)

##### 5.2.3. Unit Test: Implement Query Monitoring

In [44]:
print(f"Query ID: {invoices_bronze_query.id}")
print(f"Query Name: {invoices_bronze_query.name}")
print(f"Query Status: {invoices_bronze_query.status}")

Query ID: 96839f3b-435c-4474-92e4-31ec11fcc99e
Query Name: invoices_bronze
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [45]:
invoices_bronze_query.awaitTermination()

#### 5.3. Create the Silver Layer: Integrate "Cold-path" Data & Make Transformations
##### 5.3.1. Prepare Role-Playing Dimension Primary and Business Keys

In [46]:
df_dim_invoice_date = df_dim_date.select(col("date_key").alias("invoice_date_key"), col("full_date").alias("invoice_full_date"))

##### 5.3.2. Define Silver Query to Join Streaming with Batch Data

In [47]:
df_invoices_silver = spark.readStream.format("parquet").load(invoices_output_bronze) \
    .withColumnRenamed("InvoiceId", "invoice_id") \
    .withColumnRenamed("CustomerId", "customer_id") \
    .withColumnRenamed("BillingAddress", "billing_address") \
    .withColumnRenamed("BillingCity", "billing_city") \
    .withColumnRenamed("BillingState", "billing_state") \
    .withColumnRenamed("BillingCountry", "billing_country") \
    .withColumnRenamed("BillingPostalCode", "billing_postal_code") \
    .withColumnRenamed("Total", "invoice_total") \
    .withColumnRenamed("InvoiceLineId", "invoice_line_id") \
    .withColumnRenamed("TrackId", "track_id") \
    .withColumnRenamed("UnitPrice", "invoice_unit_price") \
    .withColumnRenamed("Quantity", "quantity") \
    .withColumn("invoice_date", date_format((col("InvoiceDate") / 1000).cast(TimestampType()), "yyyy-MM-dd")) \
    .withColumn("invoice_date_for_join", to_date((col("InvoiceDate") / 1000).cast(TimestampType()))) \
    .join(df_dim_customers, "customer_id", "inner") \
    .join(df_dim_track, "track_id", "inner") \
    .join(df_dim_invoice_date, 
          df_dim_invoice_date.invoice_full_date.cast(DateType()) == col("invoice_date_for_join").cast(DateType()), 
          "inner") \
    .withColumn("line_total", col("invoice_unit_price") * col("quantity")) \
    .select(
        col("invoice_id").cast(LongType()),
        col("invoice_line_id").cast(LongType()),
        col("invoice_date"),
        df_dim_invoice_date.invoice_date_key.cast(LongType()),
        df_dim_customers.customer_key.cast(LongType()),
        df_dim_track.track_key.cast(LongType()),
        col("quantity").cast(IntegerType()),
        col("invoice_unit_price").alias("unit_price").cast(DoubleType()),
        col("line_total").cast(DoubleType()),
        col("invoice_total").cast(DoubleType()),
        col("receipt_time"),
        col("source_file")
    )

In [48]:
df_invoices_silver.isStreaming

True

In [49]:
df_invoices_silver.printSchema()

root
 |-- invoice_id: long (nullable = true)
 |-- invoice_line_id: long (nullable = true)
 |-- invoice_date: string (nullable = true)
 |-- invoice_date_key: long (nullable = true)
 |-- customer_key: long (nullable = false)
 |-- track_key: long (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- invoice_total: double (nullable = true)
 |-- receipt_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



##### 5.3.3. Write the Transformed Streaming data to the Data Lakehouse

In [50]:
invoices_checkpoint_silver = os.path.join(invoices_output_silver, '_checkpoint')

invoices_silver_query = (
    df_invoices_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("invoices_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", invoices_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(invoices_output_silver)
)

##### 5.3.4. Unit Test: Implement Query Monitoring

In [51]:
print(f"Query ID: {invoices_silver_query.id}")
print(f"Query Name: {invoices_silver_query.name}")
print(f"Query Status: {invoices_silver_query.status}")

Query ID: 9b13a005-ff66-4682-91df-bfb07b1b0aaa
Query Name: invoices_silver
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [52]:
invoices_silver_query.awaitTermination()

#### 5.4. Create Gold Layer: Product Performance Analytics
##### 5.4.1. Define a Query to Analyze Top Tracks by Revenue and Genre Performance by Fiscal Quarter
Create Gold tables that provide insights into product performance across different dimensions.

In [53]:
# Query 1: Top Tracks by Revenue with Album and Artist Information
df_top_tracks_by_revenue = spark.readStream.format("parquet").load(invoices_output_silver) \
.join(df_dim_track, "track_key") \
.join(df_dim_date, df_dim_date.date_key.cast(IntegerType()) == col("invoice_date_key").cast(IntegerType())) \
.groupBy(
    col("track_name"),
    col("album_title"),
    col("artist_name"),
    col("genre_name")
) \
.agg(
    sum("line_total").alias("total_revenue"),
    count("invoice_line_id").alias("units_sold"),
    avg(col("line_total") / col("quantity")).alias("avg_price")
) \
.orderBy(desc("total_revenue"))

In [54]:
df_top_tracks_by_revenue.printSchema()

root
 |-- track_name: string (nullable = true)
 |-- album_title: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- units_sold: long (nullable = false)
 |-- avg_price: double (nullable = true)



##### 5.4.2. Define Query for Genre Performance by Fiscal Quarter

In [55]:
# Query 2: Genre Performance by Fiscal Quarter
df_genre_fiscal_performance = spark.readStream.format("parquet").load(invoices_output_silver) \
.join(df_dim_track, "track_key") \
.join(df_dim_date, df_dim_date.date_key.cast(IntegerType()) == col("invoice_date_key").cast(IntegerType())) \
.groupBy(
    col("fiscal_year"),
    col("fiscal_quarter"),
    col("fiscal_year_qtr"),
    col("genre_name")
) \
.agg(
    sum("line_total").alias("total_revenue"),
    count("invoice_line_id").alias("tracks_sold"),
    approx_count_distinct("customer_key").alias("unique_customers")
) \
.orderBy(asc("fiscal_year"), asc("fiscal_quarter"), desc("total_revenue"))

In [56]:
df_genre_fiscal_performance.printSchema()

root
 |-- fiscal_year: integer (nullable = true)
 |-- fiscal_quarter: byte (nullable = true)
 |-- fiscal_year_qtr: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- tracks_sold: long (nullable = false)
 |-- unique_customers: long (nullable = false)



##### 5.4.3. Write the Streaming Data to Memory in "Complete" Mode

In [57]:
top_tracks_query = (
    df_top_tracks_by_revenue.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("top_tracks_by_revenue") \
    .start()
)

In [58]:
genre_fiscal_query = (
    df_genre_fiscal_performance.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("genre_fiscal_performance") \
    .start()
)

In [59]:
wait_until_stream_is_ready(top_tracks_query, 1)
wait_until_stream_is_ready(genre_fiscal_query, 1)

The stream has processed 1 batchs
The stream has processed 1 batchs


##### 5.4.4. Query the Gold Data from Memory and Create Final Selections

In [60]:
df_top_tracks = spark.sql("SELECT * FROM top_tracks_by_revenue")
df_top_tracks.printSchema()

root
 |-- track_name: string (nullable = true)
 |-- album_title: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- units_sold: long (nullable = false)
 |-- avg_price: double (nullable = true)



In [61]:
df_genre_fiscal = spark.sql("SELECT * FROM genre_fiscal_performance")
df_genre_fiscal.printSchema()

root
 |-- fiscal_year: integer (nullable = true)
 |-- fiscal_quarter: byte (nullable = true)
 |-- fiscal_year_qtr: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- tracks_sold: long (nullable = false)
 |-- unique_customers: long (nullable = false)



##### 5.4.5. Create the Final Selections with User-Friendly Column Names

In [62]:
# Create final selection for top tracks by revenue (Top 50)
df_top_tracks_final = spark.sql("""
    SELECT 
        track_name AS Track,
        album_title AS Album,
        artist_name AS Artist,
        genre_name AS Genre,
        total_revenue AS `Total Revenue`,
        units_sold AS `Units Sold`,
        avg_price AS `Avg Price`
    FROM top_tracks_by_revenue
    ORDER BY total_revenue DESC
    LIMIT 50
""")

In [63]:
# Create final selection for genre fiscal performance
df_genre_fiscal_final = spark.sql("""
    SELECT 
        fiscal_year_qtr AS `Fiscal Quarter`,
        genre_name AS Genre,
        total_revenue AS `Total Revenue`,
        tracks_sold AS `Tracks Sold`,
        unique_customers AS `Unique Customers`
    FROM genre_fiscal_performance
    ORDER BY fiscal_year, fiscal_quarter, total_revenue DESC
""")

##### 5.4.6. Save Results to Tables and Display
Save both analytical outputs to permanent tables in the data lakehouse.

In [64]:
# Save to table and display results
df_top_tracks_final.write.saveAsTable(f"{dest_database}.top_tracks_by_revenue_final", mode="overwrite")
df_top_tracks_final.toPandas()

,Track,Album,Artist,Genre,Total Revenue,Units Sold,Avg Price
0,The Woman King,"Battlestar Galactica, Season 3",Battlestar Galactica,Science Fiction,3.98,2,1.99
1,Gay Witch Hunt,"The Office, Season 3",The Office,TV Shows,3.98,2,1.99
2,Walkabout,"Lost, Season 1",Lost,TV Shows,3.98,2,1.99
3,Phyllis's Wedding,"The Office, Season 3",The Office,Comedy,3.98,2,1.99
4,Hot Girl,"The Office, Season 1",The Office,TV Shows,3.98,2,1.99
5,How to Stop an Exploding Man,"Heroes, Season 1",Heroes,Drama,3.98,2,1.99
6,Pilot,Aquaman,Aquaman,TV Shows,3.98,2,1.99
7,The Fix,"Heroes, Season 1",Heroes,Drama,3.98,2,1.99
8,"Through the Looking Glass, Pt. 1","Lost, Season 3",Lost,Drama,1.99,1,1.99
9,"Battlestar Galactica, Pt. 1","Battlestar Galactica (Classic), Season 1",Battlestar Galactica (Classic),Sci Fi & Fantasy,1.99,1,1.99


In [65]:
# Save to table and display results
df_genre_fiscal_final.write.saveAsTable(f"{dest_database}.genre_fiscal_performance_final", mode="overwrite")
df_genre_fiscal_final.toPandas()

,Fiscal Quarter,Genre,Total Revenue,Tracks Sold,Unique Customers
0,2021Q2,Rock,1.98,2,1
1,2021Q3,Latin,32.67,33,9
2,2021Q3,Rock,30.69,31,8
3,2021Q3,Alternative & Punk,14.85,15,5
4,2021Q3,Jazz,12.87,13,6
...,...,...,...,...,...
207,2026Q2,Metal,3.96,4,2
208,2026Q2,Blues,2.97,3,2
209,2026Q2,Science Fiction,1.99,1,1
210,2026Q2,Jazz,1.98,2,1


##### 5.4.7. Analysis Insights

The Gold layer now provides two complementary views of product performance:

**1. Top Tracks by Revenue Analysis:**
- Identifies the highest revenue-generating tracks with complete metadata (track, album, artist, genre)
- Shows both revenue metrics and sales volume (units sold)
- Calculates average price per track to understand pricing dynamics
- Limited to top 50 tracks for focused executive reporting

**2. Genre Performance by Fiscal Quarter:**
- Leverages the **dim_date** dimension to analyze trends by **fiscal quarters**
- Tracks revenue, sales volume, and customer engagement (unique customers) by genre
- Enables year-over-year and quarter-over-quarter comparisons
- Supports strategic planning aligned with fiscal reporting periods

**Key Benefits:**
- Properly incorporates the dim_date dimension with fiscal period attributes
- Provides actionable insights for inventory management and marketing campaigns
- Enables data-driven decisions on genre investments and artist partnerships
- Supports both tactical (top tracks) and strategic (fiscal trends) decision-making